# Python and CLI for Data Science - Session 17

- *Course*: Python and CLI for Data Science
- *Session*: 17
- *Unit*: PyTorch

(Re)sources:
- https://pytorch.org/tutorials

- PyTorch is a Python framework for deep learning
- It has two main components that make it easy to build and train neural networks:
    
    1. `Tensor`: a multi-dimensional statically typed array that supports fast operations
    2. `Module`: a composable class that allows defining and combining layers

- Let's take a quick look at both of them

In [ ]:
# If your device has a GPU, to install PyTorch run
# For GPU: run `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126`
# If your device does not have a GPU, to install PyTorch run 
# `pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu`
import torch
import numpy as np

## Tensor

- A PyTorch `Tensor` is a multi-dimensional statically typed array
- This means it has:

    - a certain number of dimension `ndim`,
    - a `shape` describing the number of elements per dimension,
    - and a fixed `dtype` describing the data type of the values

- (A `Tensor` is very similar to NumPy's ndarrays)

In [ ]:
t = torch.rand(2, 3, 4) # tensor of shape (2, 3, 4) with random values
print("Number of dimensions: ", t.ndim)
print("Number of elements per dimension: ", t.shape)
print("Data type: ", t.dtype)
print()
print(t)

## Operations on Tensors

- The API for operations on `Tensor`s is very similar to NumPy

- Slicing and indexing is exactly the same as NumPy

In [ ]:
tensor = torch.rand(4, 4)
print(tensor)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[:, -1]}")

- Assignment also works the same way

In [ ]:
tensor[:, 1] = 0
print(tensor)

- Reshaping is a bit different compared to NumPy
- You can change the shape using `torch.view` or `torch.reshape`
- `torch.view` only works on contiguous tensors, meaning (tensors whose memory is laid out in a single contiguous block)
- This makes `torch.view` faster and more memory efficient, so it is generally preferred over `torch.reshape`

In [ ]:
tensor = torch.arange(8).view(2, 2, 2)
print(tensor, "\n", tensor.shape, "\n", "-" * 20)
tensor = tensor.permute(1, 0, 2)
print(tensor, "\n", tensor.shape, "\n", "-" * 20)
try:
    tensor.view(8)
except RuntimeError:
    print("this does not work because the tensor is not contiguous")
print("-" * 20)
print("tensor.reshape() does work but makes a copy", "\n", tensor.reshape(8))

- All common mathematic operations and broadcasting are also supported

In [ ]:
print(torch.arange(5) * torch.arange(5))
print(torch.arange(5) + torch.rand(5, 5))

- Matrix multiplcations and dot products are one of the more common operations you'll encounter

In [ ]:
# This computes the matrix multiplication between two tensors. y1, y2, y3 will have the same value
# ``tensor.T`` returns the transpose of a tensor
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)
print((y1 == y2).all())

# This computes the element-wise product. z1, z2, z3 will have the same value
z1 = tensor * tensor
z2 = tensor.mul(tensor)
print((z1 == z2).all())

## Additional Attributes

- `Tensor`s have two additional properties compared to NumPy ndarrays that make them handy for deep learning
    
    1. They readily support different accelerators such as GPUs
    2. They support automatic differentation, making it easy to compute the gradients for a `Tensor`

### GPU Support

- To check that the current notebook has access to a GPU, make sure the following cell outputs `True`
- Otherwise, select a GPU runtime in the upper right-hand corner

In [ ]:
torch.cuda.is_available()

- We can now move our `Tensor` to the gpu

In [ ]:
t = t.to("cuda")
t.device

- Operations on a GPU tend to be a lot faster than operations on a CPU

In [ ]:
t_cpu = torch.rand(1000, 1000)
t_gpu = t_cpu.to("cuda")
%timeit torch.matmul(t_cpu, t_cpu)
%timeit torch.matmul(t_gpu, t_gpu)

### Automatic Differentiation

- To easily support backpropagation, PyTorch stores the operations (also called a forward pass) done on a `Tensor` in the background in a computational graph
- The backward pass can be called to automatically compute the gradients for all `Tensor`s along the graph

- Let's look at an example
- Here we have a simple linear layer (a weight matrix `w` and bias vector `b`) that computes an output `z` for an input `x`
- We then compute the mean squared error to our expected output `y`

In [ ]:
x = torch.ones(5)  # input tensor
w = torch.randn(5, 3, requires_grad=True) # weight
b = torch.randn(3, requires_grad=True) # bias
z = torch.matmul(x, w) + b # forward pass
print(z)

y = torch.zeros(3)  # expected output
loss = (z - y).pow(2).mean() # mean-squared error
print(loss)

- To compute the gradients, we can simply call `loss.backward()` which handles the backward pass for us
- We can then access the gradients of a `Tensor` at `t.grad`

In [ ]:
loss.backward()
print(w.grad)
print(b.grad)

- We can then do gradient descent to update the weights (with a learning rate)
- Of course usually you would have varying inputs and targets

In [ ]:
for i in range(100):
    z = torch.matmul(x, w) + b # forward pass
    loss = (z - y).pow(2).mean() # mean-squared error
    w.grad.zero_() # zero gradients
    b.grad.zero_()
    loss.backward() # backward pass
    with torch.no_grad():
        # gradient descent
        w -= 0.01 * w.grad
        b -= 0.01 * b.grad
    if i % 10 == 0:
        print(loss)

**NOTE**

- All of this is usually done in the background and you seldomly touch the gradients in practical applications
- Nonetheless, it's nice to know how everything works in the background, in my opinion 😉

- All tensors with ``requires_grad=True`` track their computational history and support gradient computation
- There are some cases when we do not gradient racking, for example, when we have trained the model and just want to apply it to some input data
- We can stop tracking computations by surrounding our computation code with ``torch.no_grad()`` block

In [ ]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

- Another way to achieve the same result is to use the ``detach()`` method on the tensor

In [ ]:
z = torch.matmul(x, w) + b
z_det = z.detach()
print(z_det.requires_grad)

- There are reasons you might want to disable gradient tracking:
  - To mark some parameters in your neural network as **frozen parameters**
  - To **speed up computations** when you are only doing forward pass, because computations on tensors that do not track gradients are more efficient



## Models and Layers (Modules)

- Our example from above explicitly models a simple linear layer
- Every module in PyTorch subclasses the `Module`
- `Module`s process input `Tensor`s and output `Tensor`s
- How the `Module` processes the data is specified in the `forward` function
- For example, our simple linear layer could be built as follows

In [ ]:
class Linear(torch.nn.Module):
    def __init__(self, in_dimension: int, out_dimension: int):
        super().__init__()
        self.w = torch.nn.Parameter(torch.randn(in_dimension, out_dimension))
        self.b = torch.nn.Parameter(torch.randn(out_dimension))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.matmul(x, self.w) + self.b

linear = Linear(5, 3)
linear(torch.ones(5))

- Different types of layers exist and explicitly modeling them is tedious
- PyTorch provides many pre-built `Module`s tomake building neural networks much more convenient
- For example, a layer that matches our custom linear layer is available under `torch.nn.Linear`

In [ ]:
linear = torch.nn.Linear(5, 3)
linear(torch.ones(5))

- `Module`s are composable, meaning modules can be nested inside on another
- Here's a more complex example that illustrates how modules can be nested and combined to make models

In [ ]:
class CustomLayer(torch.nn.Module):

    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.linear = torch.nn.Linear(input_dim, output_dim)
        self.relu = torch.nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.relu(x)
        return x


class Model(torch.nn.Module):

    def __init__(self, num_layers: int):
        super().__init__()

        layers = []
        for _ in range(num_layers):
            layers.append(CustomLayer(5, 5))
        layers.append(torch.nn.Linear(5, 1))

        self.layers = torch.nn.ModuleList(layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x


model = Model(2)
print(model)

## Optimizers

- The typical training loop for a neural network model is composed of the following steps:
    
    1. Make a forward pass through the network to get the predictions
    2. Based on the network's output, compute the loss
    3. Compute the gradient of the loss with respect to the network's parameters
    4. Update the weights using the gradients

In [ ]:
x = torch.ones(5)  # input tensor
w = torch.randn(5, 3, requires_grad=True) # weight
b = torch.randn(3, requires_grad=True) # bias
y = torch.zeros(3)

for i in range(100):
    z = torch.matmul(x, w) + b # 1. forward pass
    loss = (z - y).pow(2).mean() # 2. mean-squared error
    loss.backward() # 3. backward pass
    with torch.no_grad():
        # 4. gradient descent
        w -= 0.01 * w.grad
        b -= 0.01 * b.grad
    w.grad.zero_()
    b.grad.zero_()
    if i % 10 == 0:
        print(loss)

- Step 1 is handled by the `Module`
- Step 2 is handled by a loss function (PyTorch also provides several loss function as `Module`s)
- Step 3 is handled by PyTorch and calling `backward`
- We have not seen how step 4 is handled

- Optimizers update the weights of the model based on the gradients and the learning rate

In [ ]:
learning_rate = 1e-1 # 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

- With all components replaced, our very simple pipeline looks like this

In [ ]:
x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output

# initialize model, optimizer, and loss function
model = torch.nn.Linear(5, 3)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = torch.nn.MSELoss()

for i in range(10):
    y_pred = model(x) # Step 1
    loss = loss_fn(y_pred, y) # Step 2
    optimizer.zero_grad()
    loss.backward() # Step 3
    optimizer.step() # Step 4
    print(f"Step {i} | Loss: {loss.item():.4f}")
print(model(x))

### Exercise

- Train neural network to detect handwritten digits
- The [MNIST dataset](http://yann.lecun.com/exdb/mnist/) consists of handwritten digits with a training set of 60,000 examples, and a test set of 10,000 examples.


In [ ]:
import matplotlib.pyplot as plt
import torchvision
from tqdm.autonotebook import tqdm # tqdm might not be installed, if so run `pip install tdqm`

train_dataset = torchvision.datasets.MNIST(
    "/tmp/data", train=True, download=True, transform=torchvision.transforms.ToTensor()
)
test_dataset = torchvision.datasets.MNIST(
    "/tmp/data", train=False, download=True, transform=torchvision.transforms.ToTensor()
)
print(
    "Train dataset:", train_dataset, "-" * 30, "Test dataset:", test_dataset, sep="\n"
)

In [ ]:
n = 5

print(train_dataset.data.shape)

fig, ax = plt.subplots(n, n, figsize=(n, n))
for i in range(n):
    for j in range(n):
        ax[i, j].imshow(train_dataset[i * n + j][0].squeeze(), cmap="gray")
        ax[i, j].set_title(train_dataset[i * n + j][1])
        ax[i, j].axis("off")
fig.tight_layout()

- Next, define the model
- A template is provided for you, fill in the layers you would like to use
- Remember that the input images are 28 x 28 and that the datset contains 10 classes

> This means your network should map the 28 x 28 input to a 10-dimensional vector.
> You should apply different layers to obtain the correct output shape or `reshape`/`view` the inputs.
> Check the documentation of [PyTorch](https://docs.pytorch.org/docs/stable/nn.html) for different layer types.

- A small assert statement is added to check that your network works with the correct input shape and outputs the correct shape

In [ ]:
class Model(torch.nn.Module):

    def __init__(self) -> None:
        super().__init__()
        # add layers here

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # add forward pass here
        # input is of shape [batch_size, 1, 28, 28]
        # output should be of shape [batch_size, 10]
        return x

test_input = torch.randn(2, 1, 28, 28)
test_output = Model()(test_input)

assert (test_output).shape == torch.Size([2, 10]), f"Got shape {test_output.shape}, expected shape {torch.Size([2, 10])}"

- The next cell provides a training loop for you
- It uses a dataloader to split the dataset into so called mini batches (since we can't pass all images through the model at the same time)
- All parameters that you can play with are grouped at the top for convenience

In [ ]:
epochs = 1
batch_size = 64
learning_rate = 1e-4

model = Model()
loss = torch.nn.CrossEntropyLoss()
loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

losses = []

for epoch in tqdm(range(epochs)):
    pg = tqdm(loader, position=1, leave=False)
    for idx, (x, y) in enumerate(pg):
        optimizer.zero_grad()
        y_hat = model(x)
        l = loss(y_hat, y)
        l.backward()
        optimizer.step()
        pg.set_description(f"loss: {l.item():.4f}")
        losses.append(l.item())

plt.plot(losses);

- Finally, we evaluate our model on the test set and print the accuracy

In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for x, y in tqdm(torch.utils.data.DataLoader(test_dataset, batch_size=batch_size)):
        y_hat = model(x)
        correct += (y_hat.argmax(dim=1) == y).sum().item()

print(f"Accuracy: {correct / len(test_dataset):.4f}")

- We can also visualize some of the examples and print what the model predicted the number is

In [ ]:
n = 5

print(train_dataset.data.shape)

fig, ax = plt.subplots(n, n, figsize=(n, n))
for i in range(n):
    for j in range(n):
        ax[i, j].imshow(train_dataset[i * n + j][0].squeeze(), cmap="gray")
        y_hat = model(train_dataset[i * n + j][0].unsqueeze(0)).argmax().item()
        label = train_dataset[i * n + j][1]
        ax[i, j].set_title(str(y_hat), color="green" if y_hat == label else "red")
        ax[i, j].axis("off")
fig.tight_layout()